# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">6주차 · 상품 정보를 섞어 보고 위험도 상한으로 추천을 거르기</mark>

지금까지 모델은 **누가 무엇을 담았는지(거래 기록)만** 봤습니다.

종목이 레버리지인지, 위험도가 몇인지는 **한 번도 보지 않았습니다.**

오늘은 종목에 붙은 정보인 **상품 정보(item feature)** 를 씁니다. 쓰는 방법은 두 가지입니다.

1. **점수에 섞는다** — 상품 정보 점수를 지난주 점수에 더해 본다
2. **위험도 상한을 건다** — 위험도가 투자 성향에 맞지 않는 종목을 추천 목록에서 뺀다

---

### 오늘의 구성

| 파트 | 어디서 | 하는 일 |
|---|---|---|
| 개념 | 슬라이드 | 상품 정보 · 원핫 인코딩 · 적합성원칙 · FM·DeepFM |
| 실습 | **이 노트북** | 상품 정보를 섞어 보고, 투자 성향별 위험도 상한을 정해 본다 |
| 마무리 | 슬라이드 | 오늘의 용어 · 점수판 · 다음 주 |

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기 · 이 노트북을 여는 법</mark>

1. https://colab.research.google.com/github/welovecherry/recsys/blob/main/notebooks/06_item_features.ipynb
2. 구글 계정으로 로그인합니다.
3. **경고창이 뜨면 `Run anyway` 를 누릅니다.**
4. **아래 셀의 ▶ 버튼을 누릅니다.** 실습 자료를 받아옵니다. 10초쯤 걸립니다.
5. 그다음부터는 위에서 아래로 셀을 하나씩 실행하면 됩니다.

> ⚠ **고친 내용을 남기려면** 메뉴에서 `파일 → 드라이브에 사본 저장` 을 눌러 주세요.

---

**이 노트북이 쓰는 것**

| 무엇 | 쓰임 |
|---|---|
| `data/items.csv` | 종목 100개 — 이름 · 섹터 · 자산군 · **위험도** · 테마 |
| `data/users.csv` | 투자자 300명 — **투자 성향**(안정형 · 성장형 · 공격형) |
| `data/interactions.csv` | 거래 기록 8,086건 |
| `recsys.py` | 데이터 읽기 · 시간순 분할 · 채점 · 사용자-아이템 행렬 |

⚠ **지난주 노트북을 먼저 돌리지 않아도 됩니다.** 이 노트북 하나로 끝납니다.

---

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">먼저 이 셀부터 실행하세요  ▶</mark>

In [1]:
# 이 셀에서 하는 일 — Colab 에서 열었으면 실습 자료를 내려받는다
# 왜 하나 — Colab 은 열 때마다 빈 컴퓨터라 데이터·recsys.py 가 없다. 내 컴퓨터면 그냥 넘어간다

import os          # 폴더를 만들고 옮겨 다니는 도구
import sys         # 지금 파이썬이 어떤 환경인지 알려 주는 도구
import subprocess  # 터미널 명령을 파이썬에서 대신 실행해 주는 도구

if "google.colab" in sys.modules:                    # Colab 이면 이 안이 실행된다
    if os.path.exists("/content/recsys"):            # 전에 받아 둔 것이 있으면 최신으로
        subprocess.run(["git", "-C", "/content/recsys", "pull", "-q", "--ff-only"])
    else:                                            # 처음이면 통째로 내려받는다
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/welovecherry/recsys.git", "/content/recsys"])
    os.chdir("/content/recsys/notebooks")            # 노트북 폴더 안으로 이동
    print("준비 끝 —", os.getcwd(), "· 아래 셀부터 차례로 실행하세요.")
else:
    print("내 컴퓨터에서 실행 중입니다 —", os.getcwd(), "· 따로 받아올 것이 없습니다.")

내 컴퓨터에서 실행 중입니다 — /Users/hong/workspaces/org_physical-spark/course-recsys/notebooks · 따로 받아올 것이 없습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">오늘 나오는 말 여덟 개</mark>

**1~2번은 재료, 3~6번은 추천 방식, 7~8번은 금융 규칙입니다.**

1. **상품 정보 (item feature)** — 거래 기록이 아니라 **종목 자체에 붙은 정보**
   - 오늘 쓰는 것은 `items.csv` 의 열(column) 네 개 — 섹터 · 자산군 · 위험도 · 테마
   - 2주차의 **피처(feature)** 와 같은 말입니다. 그때는 사람에게 붙은 피처(주력 섹터 · 평균 위험도)였고, 오늘은 종목에 붙은 피처입니다

2. **원핫 인코딩 (one-hot encoding)** — 글자로 된 범주를 **「그 범주면 1, 아니면 0」인 열 여러 개**로 바꾸는 방법
   - 이유 — 글자(`레버리지`)로는 곱하거나 평균을 낼 수 없습니다
   - 범주 하나마다 열이 하나 생기고, 한 줄에서 **1 이 들어가는 열은 하나뿐**이라서 「원핫(one-hot)」이라고 부릅니다
   - 예 — 종목 세 개의 섹터를 원핫 인코딩하면 아래처럼 바뀝니다. 자세한 설명은 **6.3** 에 있습니다

**변환 전** — 섹터가 글자

| 종목 | 섹터 |
|---|---|
| KODEX 레버리지 | 레버리지 |
| 삼성생명 | 금융 |
| Procter & Gamble | 소비재 |

**변환 후** — 섹터마다 열이 하나. 그 섹터면 1, 아니면 0

| 종목 | 레버리지 | 금융 | 소비재 |
|---|---|---|---|
| KODEX 레버리지 | **1** | 0 | 0 |
| 삼성생명 | 0 | **1** | 0 |
| Procter & Gamble | 0 | 0 | **1** |


3. **협업 필터링 (collaborative filtering)** — **다른 사람들의 거래 기록**으로 추천하는 방식
   - 「나와 비슷하게 담은 사람들이 담은 종목」을 추천합니다
   - 4주차 행렬분해가 여기에 속합니다. 이 방식으로 낸 점수를 오늘은 **협업 필터링 점수**라고 부릅니다

4. **콘텐츠 기반 추천 (content-based filtering)** — **상품 정보**로 추천하는 방식
   - 「내가 담아 온 종목과 상품 정보가 비슷한 종목」을 추천합니다
   - 사용자가 담아 온 종목의 상품 정보를 모은 것을 **사용자 프로필(user profile)** 이라고 합니다
   - 다른 사람의 기록은 보지 않습니다. 노트북 **6.4** 에서 다룹니다

5. **하이브리드 추천 (hybrid recommendation)** — 3번과 4번의 점수를 **섞어서** 추천하는 방식
   - 섞는 비율을 **α(알파)** 라고 부릅니다. 노트북 **6.5** 에서 다룹니다

6. **표준화 (standardization)** — 점수에서 **평균을 빼고 표준편차로 나누는** 계산
   - 두 점수는 범위가 달라서, 섞기 전에 **범위를 똑같이 맞춥니다**
   - **표준편차(standard deviation)** — 값이 평균에서 보통 얼마나 떨어져 있는지 나타내는 수
   - 예 — `1, 2, 3` 을 표준화하면 `-1.22, 0, 1.22` (평균인 2 가 0 이 됩니다)

7. **적합성원칙 (suitability rule)** — 금융 회사는 **고객의 투자 성향에 맞지 않는 상품을 권유하면 안 된다**는 원칙
   - 금융소비자보호법 제17조에 있습니다
   - **투자 성향**은 고객이 가입할 때 설문으로 정합니다 — 우리 데이터에서는 안정형 · 성장형 · 공격형

8. **위험도 상한 (risk cap)** — 투자 성향마다 정한 **추천해도 되는 위험도의 최댓값**
   - 이 값을 넘는 종목은 추천 목록에서 뺍니다. 노트북 **6.7 ~ 6.8**
   - 모델이 만든 목록에서 규칙으로 종목을 빼는 것을 **후처리 필터링(post-filtering)** 이라고 합니다

---

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">준비 — 데이터 읽기와 시간순 분할</mark>

**여기서 하는 일** — 데이터를 읽고, 3주차부터 쓰던 **시간순 분할**로 학습 구간과 채점 구간을 나눕니다.

**채점 기준이 같아야** 오늘 숫자를 4주차 **0.3918** 과 비교할 수 있습니다.

In [2]:
# 이 셀에서 하는 일 — 도구를 불러오고, 데이터를 읽고, 시간순으로 나눈다
# 왜 하나 — 지난주와 같은 기준으로 채점해야 0.3918 과 비교할 수 있다

import sys                                  # 파이썬이 파일을 찾는 경로를 다루는 도구
sys.path.insert(0, ".")                     # 지금 폴더에서 recsys.py 를 찾게 한다
sys.path.insert(0, "notebooks")             # 한 단계 안쪽 폴더(notebooks)도 찾게 한다

import numpy as np                          # 숫자 묶음을 빠르게 다루는 도구. np 라고 부른다
import pandas as pd                         # 표를 다루는 도구. pd 라고 부른다
import recsys                               # 이 수업용 도구 모음 — 읽기·분할·채점이 들어 있다

items, users, interactions = recsys.load()                  # 종목·투자자·거래 기록 세 표
train, test, 기준시점 = recsys.split_by_time(interactions)   # 3주차부터 쓰던 시간순 분할

종목이름 = items.set_index("item_id")["name"].to_dict()        # to_dict = 딕셔너리로. {종목 번호: 이름}
이미_담은것 = train.groupby("user_id")["item_id"].apply(set).to_dict()   # groupby = 사람별로 묶는다
전체_인기순위 = list(train["item_id"].value_counts().index)   # value_counts = 같은 값이 몇 번인지 센다

print(f"학습 {len(train):,}건 · 채점 {len(test):,}건 · 자른 날짜 {기준시점.date()}")

학습 7,095건 · 채점 991건 · 자른 날짜 2026-07-30


**결과 읽는 법**

- 거래 기록 8,086건을 2026-07-30 을 기준으로 둘로 나눈 결과입니다.
  - **학습 7,095건**은 자른 날짜 앞쪽 기록입니다. 모델이 배우는 데 씁니다.
  - **채점 991건**은 자른 날짜 뒤쪽 기록입니다. 추천이 맞았는지 확인하는 데 씁니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.1 준비 — 4주차 행렬분해를 다시 만든다</mark>

**6.1 에서 하는 일** — 4주차의 **사용자 요인(user factor, 285 × 8)** 과 **아이템 요인(item factor, 8 × 100)** 을 다시 만들고, 종목 100개 전부에 점수를 매기는 4주차 추천을 다시 돌립니다.

**1. 왜 5주차가 아니라 4주차와 비교하나**
- 오늘 비교할 대상은 **4주차 0.3918** 입니다.
- 5주차 2단계 추천까지 섞으면, 점수가 바뀐 것이 상품 정보 때문인지 후보 40개 때문인지 가릴 수 없습니다.
- **한 번에 하나만 바꿔야** 비교가 정확합니다.

**2. 함수 두 개를 만듭니다**
- `점수순으로_종목_정렬` — 종목 100개의 점수를 받아 **점수가 높은 종목부터 종목 번호를 차례로 담은 목록**을 돌려줍니다. 오늘 만드는 추천 네 가지가 모두 이 함수를 씁니다.
- `지난주_추천` — 4주차 추천과 같습니다. 사용자 요인 한 줄 × 아이템 요인 = 종목 100개 점수.

**3. 변수 이름**
- 4·5주차 노트북의 `사람표` · `종목표` 를 오늘은 공식 용어 그대로 `사용자_요인` · `아이템_요인` 이라고 부릅니다.
- 285 × 100 짜리 「담았으면 1, 아니면 0」 표는 **사용자-아이템 행렬(user-item matrix)** 이므로 `상호작용_행렬` 이라고 부릅니다.

In [3]:
# 이 셀에서 하는 일 — 4주차와 똑같이 사용자-아이템 행렬을 만들고 잠재 요인 8개로 쪼갠다
# 왜 하나 — 오늘 쓰는 모델이 지난주 것과 같아야 상품 정보의 효과만 볼 수 있다

from sklearn.decomposition import TruncatedSVD      # 4주차에 쓴 행렬분해 도구

상호작용_행렬 = recsys.interaction_matrix(train, items)        # 사용자-아이템 행렬 — 담았으면 1, 아니면 0
모델 = TruncatedSVD(n_components=8, random_state=42)   # 잠재 요인 8개 — 지난주와 같은 설정
사용자_요인 = 모델.fit_transform(상호작용_행렬)                      # 사용자 요인(user factor) · 285 × 8
아이템_요인 = 모델.components_                            # 아이템 요인(item factor) · 8 × 100

사용자_목록 = list(상호작용_행렬.index)                         # 행렬의 행 이름(투자자 번호) 목록
아이템_목록 = list(상호작용_행렬.columns)                            # 행렬의 열 이름(종목 번호) 목록

print(f"사용자_요인 {사용자_요인.shape} · 아이템_요인 {아이템_요인.shape}   ← 지난주와 같은 크기")

사용자_요인 (285, 8) · 아이템_요인 (8, 100)   ← 지난주와 같은 크기


**결과 읽는 법**

- 4주차와 같은 크기의 요인 두 개가 만들어졌습니다.
  - `사용자_요인 (285, 8)` 은 투자자 285명 × 잠재 요인 8개입니다.
  - `아이템_요인 (8, 100)` 은 잠재 요인 8개 × 종목 100개입니다.

In [4]:
# 이 셀에서 하는 일 — 종목 100개의 점수를 받아, 점수가 큰 종목부터 늘어놓는 함수를 만든다
# 왜 하나 — 오늘 만드는 추천 네 가지가 모두 「점수가 큰 종목부터 늘어놓기」를 한다

def 점수순으로_종목_정렬(점수):                       # def = 함수를 만든다. 종목 100개의 점수를 받는다
    """종목 100개의 점수를 받아, 점수가 큰 종목부터 종목 번호를 늘어놓은 목록을 돌려준다."""
    정렬된_종목 = []                                # 종목 번호를 점수 순서대로 담을 빈 목록
    for 자리 in np.argsort(-점수):                  # argsort(-점수) = 점수가 큰 것부터 그 자리 번호
        정렬된_종목.append(아이템_목록[자리])          # 자리 번호(0~99) → 종목 번호(I001~I100)
    return 정렬된_종목                              # 점수 1등 종목부터 100등 종목까지

In [5]:
# 이 셀에서 하는 일 — 방금 만든 함수를 U0001 의 점수에 한 번 써 본다
# 왜 하나 — 함수가 무엇을 돌려주는지 눈으로 확인한다

U0001_점수 = 사용자_요인[사용자_목록.index("U0001")] @ 아이템_요인   # U0001 의 종목 100개 점수 (4주차 계산)
정렬된_종목 = 점수순으로_종목_정렬(U0001_점수)                     # 점수가 큰 종목부터 늘어놓는다

순위 = 1                                                          # 몇 등인지 세는 카운터
for 번호 in 정렬된_종목[:3]:                                       # [:3] = 앞에서 3개만
    print(f"{순위}등 {번호} · 점수 {U0001_점수[아이템_목록.index(번호)]:.3f} · {종목이름[번호]}")
    순위 = 순위 + 1                                               # 다음 등수로
print("※ 세 종목 모두 U0001 이 이미 담은 종목입니다. 이것을 빼는 일은 다음 셀의 recsys.take 가 합니다.")

1등 I007 · 점수 0.973 · KODEX 국고채30년액티브
2등 I006 · 점수 0.909 · TIGER 리츠부동산인프라
3등 I019 · 점수 0.794 · Procter & Gamble
※ 세 종목 모두 U0001 이 이미 담은 종목입니다. 이것을 빼는 일은 다음 셀의 recsys.take 가 합니다.


**결과 읽는 법**

- U0001 의 4주차 점수가 높은 종목 3개입니다.
  - 한 줄은 「등수 · 종목 번호 · 점수 · 종목 이름」 순서입니다.
  - 1등은 I007 KODEX 국고채30년액티브이고, 점수는 **0.973** 입니다.
- 마지막 줄 ※ 는 이 세 종목이 U0001 이 **이미 담은 종목**이라는 뜻입니다.
  - 그래서 이 세 종목은 실제 추천 목록에는 들어가지 않습니다.

In [6]:
# 이 셀에서 하는 일 — 4주차 추천 함수를 다시 만들고 U0001 에게 한 번 돌려 본다
# 왜 하나 — 오늘 모든 비교의 출발점이다

def 지난주_추천(사람, 이미):                          # def = 함수를 만든다
    """4주차 방식 — 종목 100개 전부에 점수를 매겨 높은 순으로 10개."""
    if 사람 not in 사용자_목록:                      # 사용자-아이템 행렬에 없는 신규 투자자는
        return recsys.take(전체_인기순위, 이미)      # 전체 인기 목록으로 준다
    점수 = 사용자_요인[사용자_목록.index(사람)] @ 아이템_요인   # @ = 행렬 곱. 사용자 요인 한 줄 × 아이템 요인
    return recsys.take(점수순으로_종목_정렬(점수), 이미)        # take = 이미 담은 것을 빼고 위에서 10개

print("U0001 에게 줄 추천 1등 :", 종목이름[지난주_추천("U0001", 이미_담은것["U0001"])[0]])

U0001 에게 줄 추천 1등 : TIGER 미국배당다우존스


**결과 읽는 법**

- U0001 이 이미 담은 종목을 빼고 남은 종목 가운데 1등입니다.
  - 바로 위 셀의 1등(KODEX 국고채30년액티브)이 빠지고 **TIGER 미국배당다우존스**가 1등이 되었습니다.

In [7]:
# 이 셀에서 하는 일 — 4주차 추천을 3주차와 같은 기준으로 채점한다
# 왜 하나 — 0.3918 이 나와야 지난주 모델을 제대로 다시 만든 것이다

지난주_점수, 인원 = recsys.recall_at_k(지난주_추천, train, test)   # 3주차부터 쓰던 채점 함수

print(f"4주차 추천 — Recall@10 = {지난주_점수:.4f}   (채점 대상 {인원}명)")

4주차 추천 — Recall@10 = 0.3918   (채점 대상 266명)


**결과 읽는 법**

- 6.1 에서 다시 만든 4주차 추천의 채점 결과입니다.
  - Recall@10 은 **0.3918** 로, 4주차에 나온 값과 같습니다.
  - 채점 대상은 **266명**입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.2 4주차 추천의 위험도 — 안정형 U0001 의 10개</mark>

**6.2 에서 하는 일**
- 6.1 에서 되살린 4주차 모델로 **U0001 에게 줄 추천 10개**를 뽑습니다.
- 각 종목 옆에 그 종목의 **위험도(1 ~ 5)** 를 적어 봅니다.
- U0001 은 **안정형** 투자자입니다. 추천 10개 안에 위험도가 높은 종목이 섞여 있는지 확인합니다.

**왜 확인하나**
- 4주차 모델은 **거래 기록만** 보고 추천했습니다.
  - 종목이 얼마나 위험한지(위험도)는 보지 않았습니다.
  - 투자자가 어떤 투자 성향인지도 보지 않았습니다.
- 그래서 안정형 투자자에게 위험도가 높은 종목을 추천했을 수 있습니다. 실제로 그런지 출력으로 확인합니다.

**1. 위험도란**
- `items.csv` 의 `risk_level` 열입니다. 종목마다 1 ~ 5 중 하나가 적혀 있습니다.
  - 1 = 가장 낮음 (예: KODEX 단기채권)
  - 5 = 가장 높음 (예: KODEX 레버리지)
- 종목 100개 중 위험도 4 는 28개, 5 는 20개입니다.

**2. 투자 성향이란**
- `users.csv` 의 `persona` 열입니다. 투자자마다 안정형 · 성장형 · 공격형 중 하나가 적혀 있습니다.
  - 안정형 = 원금 손실을 되도록 피하려는 투자자
  - 성장형 = 어느 정도 손실을 감수하고 수익을 노리는 투자자
  - 공격형 = 큰 손실 가능성을 감수하고 높은 수익을 노리는 투자자
- 우리 데이터에는 세 성향이 100명씩 있습니다.
- 실제 금융 회사는 고객이 계좌를 열 때 **투자성향 설문**으로 이 값을 알아냅니다.

**3. 2주차의 세그먼트와 무엇이 다른가**
- 2주차에는 성향을 **거래 기록으로 짐작**했습니다.
  - 주력 섹터와 평균 위험도로 세그먼트(예: `레버리지|4`)를 만들었습니다.
- 오늘은 **고객이 설문에 직접 답한 성향**(`persona`)을 씁니다.
  - 「이 고객에게 위험도 몇까지 추천해도 되나」는 짐작으로 정하면 안 됩니다. 고객이 직접 밝힌 성향을 따라야 합니다.

**4. 아래 코드가 하는 일 (3단계)**
1. `성향` 딕셔너리를 만든다 — 투자자 번호 → 투자 성향 (예: U0001 → 안정형)
2. `위험도` 딕셔너리를 만든다 — 종목 번호 → 위험도 (예: I079 → 5)
3. U0001 의 추천 10개를 한 줄씩 출력한다
   - 위험도가 **4 이상**이면 줄 앞에 ⚠ 를 붙인다

**5. 출력에서 찾을 것**
- ⚠ 가 붙은 줄이 몇 개인지 세어 보세요.

In [8]:
# 이 셀에서 하는 일 — U0001 의 추천 10개에 위험도를 붙여 본다
# 왜 하나 — 지난주 모델이 투자 성향을 전혀 보지 않는다는 것을 눈으로 확인한다

성향 = users.set_index("user_id")["persona"].to_dict()       # 딕셔너리 {투자자 번호: 안정형/성장형/공격형}
위험도 = items.set_index("item_id")["risk_level"].to_dict()   # 딕셔너리 {종목 번호: 1~5}
print(f"U0001 의 투자 성향 : {성향['U0001']}")

순위 = 1                                                      # 몇 등인지 세는 카운터
for 번호 in 지난주_추천("U0001", 이미_담은것["U0001"]):      # 지난주 모델이 준 10개
    표시 = "⚠" if 위험도[번호] >= 4 else " "                 # >= 크거나 같다. 4 이상이면 ⚠
    print(f"{순위:2d}. 위험도 {위험도[번호]} {표시} {종목이름[번호]}")   # :2d = 두 자리로 오른쪽 정렬
    순위 = 순위 + 1                                           # 다음 등수로

U0001 의 투자 성향 : 안정형
 1. 위험도 2   TIGER 미국배당다우존스
 2. 위험도 5 ⚠ TIGER 2차전지테마
 3. 위험도 3   Vanguard FTSE Developed Markets ETF
 4. 위험도 2   TIGER 은행고배당플러스TOP10
 5. 위험도 1   KODEX 단기채권
 6. 위험도 4 ⚠ 크래프톤
 7. 위험도 4 ⚠ Taiwan Semiconductor
 8. 위험도 2   Verizon Communications
 9. 위험도 2   Johnson & Johnson
10. 위험도 4 ⚠ Advanced Micro Devices


**결과 읽는 법**

- ⚠ 가 **4개**입니다. 안정형 U0001 에게 위험도 4 ~ 5 종목이 10개 중 4개 들어갔습니다.
  - 2등 TIGER 2차전지테마 — 위험도 5
  - 6등 크래프톤 · 7등 Taiwan Semiconductor · 10등 Advanced Micro Devices — 위험도 4

- 왜 이런 추천이 나왔나
  - 4주차 모델은 **협업 필터링**입니다. U0001 과 **거래 기록이 비슷한 다른 투자자들**이 담은 종목을 추천합니다.
  - 그 투자자 가운데 위험도가 높은 종목을 담은 사람이 있으면, 그 종목이 U0001 의 추천에도 들어옵니다.
  - 모델은 종목의 위험도도, U0001 의 투자 성향도 모릅니다. 거래 기록만 입력으로 받았기 때문입니다.

- 그래서 오늘 두 가지를 해 봅니다
  - 6.3 ~ 6.5 — 상품 정보를 점수에 **섞으면** 추천이 나아지는지 본다
  - 6.6 ~ 6.8 — 투자 성향에 맞지 않는 위험도의 종목을 추천 목록에서 **뺀다**

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.3 원핫 인코딩 — 상품 정보를 0/1 열로 펼치기</mark>

**6.3 에서 하는 일** — 종목마다 네 가지 상품 정보를 **0과 1로 된 열**로 바꿉니다.

**1. 왜 바꾸나**
- `레버리지` · `금융` 같은 글자로는 곱셈도 평균도 못 합니다.
- 6.4 에서 「담은 종목의 평균」을 구하려면 숫자여야 합니다.

**2. 예시 — 종목 세 개의 섹터만 펼쳐 보면**

① **원래** — 섹터가 글자로 적혀 있습니다.

| 종목 | 섹터 |
|---|---|
| KODEX 레버리지 | 레버리지 |
| 삼성생명 | 금융 |
| Procter & Gamble | 소비재 |

② **원핫 인코딩 후** — 섹터 하나마다 열을 하나 만들고, 그 섹터면 1, 아니면 0 을 적습니다.

| 종목 | 레버리지 | 금융 | 소비재 |
|---|---|---|---|
| KODEX 레버리지 | **1** | 0 | 0 |
| 삼성생명 | 0 | **1** | 0 |
| Procter & Gamble | 0 | 0 | **1** |

③ **읽는 법**
- 맨 윗줄 `레버리지 · 금융 · 소비재` 는 **열 이름**, 왼쪽 종목 이름은 **행 이름**입니다. 둘 다 이름일 뿐이라 계산에는 쓰지 않습니다.
- **계산에 쓰는 값은 가운데의 0 과 1 뿐**입니다.
- 한 줄에 1 이 **딱 하나만** 있습니다. 그래서 「원핫(one-hot)」입니다.
- **바로 아래 첫 셀이 이 두 표를 그대로 만들고, 두 번째 셀은 값(0 과 1)만 꺼내 보여 줍니다.**

**3. 왜 1 · 2 · 3 으로 번호를 붙이지 않나**
- 레버리지 = 1, 금융 = 2, 소비재 = 3 으로 적으면 **「금융은 레버리지의 두 배」, 「소비재와 레버리지의 평균이 금융」** 같은 없는 뜻이 생깁니다.
- 섹터에는 크기 순서가 없습니다. 원핫 인코딩을 하면 **어느 두 범주든 서로 거리가 같습니다**.

**4. 네 가지를 다 펼치면**
- 섹터 25 + 자산군 2 + 위험도 5 + 테마 9 = **41개 열**
- 열 이름 앞에 원래 열 이름이 붙습니다 — `sector_레버리지` · `theme_레버리지`. 「레버리지」가 섹터에도 테마에도 있어서, 붙이지 않으면 두 열의 이름이 같아집니다.
- 종목마다 정보가 네 가지이므로, 한 줄에 **1 이 정확히 4개**입니다(아래 출력의 I079).
- 위험도는 숫자지만 **범주로 다룹니다.** 「위험도 4 인가 아닌가」만 봅니다.

**5. 어떤 함수를 쓰나**
- `pd.get_dummies` — pandas 의 원핫 인코딩 함수입니다. 「더미(dummy) 변수」는 0/1 열을 가리키는 통계 용어입니다.

In [9]:
# 이 셀에서 하는 일 — 종목 세 개의 섹터를 원핫 인코딩해 본다 (위 ① → ②)
# 왜 하나 — 100개 × 41열로 가기 전에, 작은 예시로 무엇이 바뀌는지 먼저 본다

섹터_예시 = pd.Series(["레버리지", "금융", "소비재"],
                      index=["KODEX 레버리지", "삼성생명", "Procter & Gamble"])   # Series = 이름이 붙은 값 목록
print(섹터_예시.to_frame("섹터"))                         # ① 원래 — to_frame = 표 모양으로 보여 준다
print()

원핫_예시 = pd.get_dummies(섹터_예시).astype(int)          # get_dummies = 원핫 인코딩. astype(int) = True/False → 1/0
원핫_예시 = 원핫_예시[["레버리지", "금융", "소비재"]]        # 열 순서를 위 설명 표와 같게 맞춘다
print(원핫_예시)                                          # ② 원핫 인코딩 후

                    섹터
KODEX 레버리지        레버리지
삼성생명                금융
Procter & Gamble   소비재

                  레버리지  금융  소비재
KODEX 레버리지           1   0    0
삼성생명                 0   1    0
Procter & Gamble     0   0    1


**결과 읽는 법**

- 위쪽 표는 원래 모양(①)입니다.
  - 종목 세 개의 섹터가 글자로 적혀 있습니다.
- 아래쪽 표는 원핫 인코딩 후(②)입니다.
  - 섹터마다 열이 하나씩 생기고, 그 섹터인 칸만 **1** 입니다.

In [10]:
# 이 셀에서 하는 일 — 열 이름 · 행 이름을 떼고 값만 꺼내 본다
# 왜 하나 — 표에 보이는 글자는 열 이름 · 행 이름일 뿐, 계산에 쓰는 것은 0 과 1 뿐이라는 것을 확인한다

print(원핫_예시.values)                                   # .values = 열 이름 · 행 이름을 떼고 값만

[[1 0 0]
 [0 1 0]
 [0 0 1]]


**결과 읽는 법**

- 바로 위 아래쪽 표에서 열 이름과 행 이름을 떼고 0 과 1 만 남긴 것입니다.
  - 세 줄은 위에서부터 KODEX 레버리지 · 삼성생명 · Procter & Gamble 입니다.
  - 세 칸은 왼쪽부터 레버리지 · 금융 · 소비재 열입니다.

In [11]:
# 이 셀에서 하는 일 — 종목 100개의 상품 정보 네 가지를 0/1 열 41개로 펼친다
# 왜 하나 — 글자로는 계산을 못 한다. 0/1 이어야 평균도 곱셈도 된다

정보_열 = ["sector", "asset_class", "risk_level", "theme"]          # 섹터 · 자산군 · 위험도 · 테마
종목_정보 = items.set_index("item_id")[정보_열].astype(str)          # astype(str) = 위험도 숫자도 글자로 본다
아이템_피처 = pd.get_dummies(종목_정보).astype(int)                       # get_dummies = 원핫 인코딩. 결과가 아이템 피처 행렬(item feature matrix)
아이템_피처 = 아이템_피처.loc[아이템_목록]                                          # loc = 종목 순서를 사용자-아이템 행렬과 맞춘다
print(f"아이템_피처 {아이템_피처.shape}   ← 종목 100개 × 열 41개")

값이_1인_열 = []                                                         # 1 인 열 이름을 담을 빈 목록
for 열 in 아이템_피처.columns:                                            # 41개 열을 하나씩 본다
    if 아이템_피처.loc["I079", 열] == 1:                                   # == 같다. 1 인 열만 모은다
        값이_1인_열.append(열)                                            # append = 목록 끝에 붙인다
print(f"I079 {종목이름['I079']} → 1 인 열 : {값이_1인_열}")

아이템_피처 (100, 41)   ← 종목 100개 × 열 41개
I079 KODEX 레버리지 → 1 인 열 : ['sector_레버리지', 'asset_class_ETF', 'risk_level_5', 'theme_레버리지']


**결과 읽는 법**

- 첫 줄 `(100, 41)` 은 아이템 피처 행렬의 크기입니다.
  - 종목 100개 × 0/1 열 41개입니다.
- 둘째 줄은 I079 KODEX 레버리지 줄에서 값이 1 인 열 이름입니다.
  - 섹터 · 자산군 · 위험도 · 테마에서 하나씩, 모두 **4개**입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.4 콘텐츠 기반 추천 — 사용자 프로필로 점수 매기기</mark>

**6.4 에서 하는 일** — 사용자가 담은 종목의 0/1 줄을 **평균**해서 **사용자 프로필(user profile)** 을 만들고, 이 프로필만으로 추천합니다. 이렇게 추천하는 방식이 **콘텐츠 기반 추천(content-based filtering)** 입니다.

**1. 사용자 프로필을 만드는 법 (2단계)**
1. 사용자-아이템 행렬(285 × 100)과 아이템 피처 행렬(100 × 41)을 곱합니다 → 사람마다 **열별로 1 이 몇 번 나왔는지** 셉니다.
2. 그 사람이 담은 종목 수로 나눕니다 → **비율**이 됩니다. 「담은 종목 가운데 몇 %가 위험도 5 인가」입니다.

**2. 콘텐츠 기반 점수를 매기는 법**
- 사용자 프로필의 값 41개와 종목의 0/1 값 41개를 같은 열끼리 곱해 더합니다. 4주차의 **내적(dot product)** 과 같은 계산입니다.
- 사용자 프로필에서 비율이 높은 열에 **1 이 많은 종목일수록** 점수가 높습니다.
- 거래 기록의 「누가 무엇을 같이 담았나」는 **전혀 쓰지 않습니다.** 상품 정보만 씁니다.

In [12]:
# 이 셀에서 하는 일 — 사용자 프로필(열 41개의 비율)을 만든다
# 왜 하나 — 「이 사용자는 어떤 상품 정보를 가진 종목을 담아 왔나」를 숫자로 만든다

담은수 = 상호작용_행렬.sum(axis=1)                       # axis=1 = 가로로 더한다 → 사람마다 담은 종목 수
사용자_프로필 = 상호작용_행렬.dot(아이템_피처)                       # dot = 행렬 곱. 사람마다 열별로 1 이 몇 번인지 센다
사용자_프로필 = 사용자_프로필.div(담은수, axis=0)           # div(axis=0) = 줄마다 그 사람이 담은 종목 수로 나눈다
print(f"사용자_프로필 {사용자_프로필.shape}   ← 사람 285명 × 열 41개")

print(f"U0001 은 {담은수['U0001']}개를 담았다. 그중 위험도별 비율 :")
for 단계 in ["1", "2", "3", "4", "5"]:        # 위험도 열 다섯 개만 본다
    print(f"  위험도 {단계} : {사용자_프로필.loc['U0001', 'risk_level_' + 단계]:.0%}")   # .0% = 정수 퍼센트

사용자_프로필 (285, 41)   ← 사람 285명 × 열 41개
U0001 은 16개를 담았다. 그중 위험도별 비율 :
  위험도 1 : 6%
  위험도 2 : 31%
  위험도 3 : 44%
  위험도 4 : 19%
  위험도 5 : 0%


**결과 읽는 법**

- 첫 줄 `(285, 41)` 은 사용자 프로필의 크기입니다.
  - 투자자 285명 × 상품 정보 열 41개입니다.
- 아래 다섯 줄은 U0001 의 사용자 프로필 가운데 위험도 열 다섯 개만 꺼낸 것입니다.
  - U0001 이 담은 종목 **16개** 가운데 위험도별로 몇 %인지를 나타냅니다.
  - 위험도 3 이 **44%** 로 가장 크고, 위험도 5 는 **0%** 입니다.

In [13]:
# 이 셀에서 하는 일 — 사용자 프로필만으로 점수를 매겨 추천하고 채점한다 (콘텐츠 기반 추천)
# 왜 하나 — 상품 정보 하나만으로는 얼마나 맞히는지 먼저 본다

콘텐츠_점수_행렬 = 사용자_프로필.dot(아이템_피처.T)               # .T = 가로세로를 바꾼다(전치). 결과는 사용자 × 종목 콘텐츠 기반 점수

def 콘텐츠_기반_추천(사람, 이미):                   # 4주차 함수와 모양이 같다. 점수만 다르다
    """콘텐츠 기반 점수가 높은 종목부터 10개."""
    if 사람 not in 사용자_목록:                  # 사용자-아이템 행렬에 없는 신규 투자자는
        return recsys.take(전체_인기순위, 이미)  # 전체 인기 목록으로 준다
    점수 = 콘텐츠_점수_행렬.loc[사람].values             # .values = 표 한 줄을 숫자 묶음으로
    return recsys.take(점수순으로_종목_정렬(점수), 이미)    # 이미 담은 것을 빼고 위에서 10개

콘텐츠만_점수, _ = recsys.recall_at_k(콘텐츠_기반_추천, train, test)   # _ = 쓰지 않을 값(인원)을 받아 두는 이름
print(f"콘텐츠 기반 추천 — Recall@10 = {콘텐츠만_점수:.4f}   (4주차 {지난주_점수:.4f})")

콘텐츠 기반 추천 — Recall@10 = 0.2463   (4주차 0.3918)


**결과 읽는 법**

- 바로 위 출력에서 U0001 은 **위험도 5 를 한 번도 담지 않았습니다(0%).** 그런데 6.2 에서 위험도 5 종목이 2등이었습니다. 4주차 모델이 U0001 본인이 아니라 **비슷한 사람들**의 기록을 따랐기 때문입니다.

- 상품 정보만 쓰면 Recall@10 이 4주차보다 **훨씬 낮습니다.** 왜 낮은지는 노트북 끝 「혼자 해 보기 1」에서 생각해 봅니다.

- 그래서 다음 절에서 협업 필터링 점수와 **둘을 섞어 봅니다.**

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.5 하이브리드 추천 — 두 점수를 α 비율로 섞기</mark>

**6.5 에서 하는 일** — 4주차의 **협업 필터링 점수**와 6.4 의 **콘텐츠 기반 점수**를 **α(알파)** 비율로 섞습니다. 이것이 **하이브리드 추천(hybrid recommendation)** 입니다.

**1. 섞는 식**
- `하이브리드 점수 = (1 - α) × 협업 필터링 점수 + α × 콘텐츠 기반 점수`
- 두 점수에 비율을 곱해 더하는 이 계산을 **가중합(weighted sum)** 이라고 합니다.
- α = 0 이면 4주차 점수 그대로이고, α = 1 이면 상품 정보만 쓰며, α = 0.1 이면 협업 90% + 상품 10% 입니다.

**2. 먼저 두 점수의 범위를 똑같이 맞춥니다 — 표준화 (standardization)**
- 두 점수는 범위가 다릅니다. 그대로 더하면 순위가 거의 범위가 넓은 쪽 점수대로 정해집니다.
- 그래서 각 점수에서 **평균을 빼고 표준편차로 나눕니다.** 그러면 두 점수 모두 평균 0, 표준편차 1 이 됩니다.

In [14]:
# 이 셀에서 하는 일 — 두 점수의 범위를 똑같이 맞추는 함수(표준화)를 만들고 한 번 써 본다
# 왜 하나 — 범위가 다른 두 점수를 그대로 더하면 순위가 거의 범위가 넓은 쪽 점수대로 정해진다

def 표준화(값):
    """평균을 빼고 표준편차로 나눈다 — 평균 0, 표준편차 1 이 된다."""
    return (값 - 값.mean()) / 값.std()          # mean = 평균, std = 표준편차

print("함수 확인 :", 표준화(np.array([1.0, 2.0, 3.0])))   # 가운데 값 2 가 0 이 된다

함수 확인 : [-1.22474487  0.          1.22474487]


**결과 읽는 법**

- `1, 2, 3` 세 값을 표준화한 결과입니다.
  - 평균인 2 는 **0** 이 되었습니다.
  - 1 은 **-1.22474487**, 3 은 **1.22474487** 이 되었습니다.

In [15]:
# 이 셀에서 하는 일 — 협업 필터링 점수와 콘텐츠 기반 점수를 α 비율로 섞는다 (하이브리드 추천)
# 왜 하나 — 상품 정보를 더하면 4주차보다 좋아지는지 보려는 것이다

알파 = 0.1                                       # 아래 셀에서 바꿔 본다

def 하이브리드_추천(사람, 이미):                        # 4주차 함수와 모양이 같다. 점수만 두 점수의 가중합
    """협업 필터링 점수와 콘텐츠 기반 점수의 가중합이 높은 순으로 10개."""
    if 사람 not in 사용자_목록:                   # 사용자-아이템 행렬에 없는 신규 투자자는
        return recsys.take(전체_인기순위, 이미)   # 전체 인기 목록으로 준다
    협업_점수 = 표준화(사용자_요인[사용자_목록.index(사람)] @ 아이템_요인)   # 협업 필터링 점수, 표준화한 것
    콘텐츠_점수 = 표준화(콘텐츠_점수_행렬.loc[사람].values)      # 콘텐츠 기반 점수, 표준화한 것
    하이브리드_점수 = (1 - 알파) * 협업_점수 + 알파 * 콘텐츠_점수    # 하이브리드 점수 = 가중합
    return recsys.take(점수순으로_종목_정렬(하이브리드_점수), 이미)  # 하이브리드 점수가 높은 순으로 10개

In [16]:
# 이 셀에서 하는 일 — α 를 바꿔 가며 채점한다
# 왜 하나 — 상품 정보를 조금 섞을 때와 많이 섞을 때를 비교한다

for 값 in [0.0, 0.1, 0.3, 0.5, 1.0]:              # 0 = 4주차 그대로, 1 = 상품 정보만
    알파 = 값                                    # 위 함수가 이 값을 보고 섞는다
    점수, _ = recsys.recall_at_k(하이브리드_추천, train, test)   # _ = 쓰지 않을 값(인원)을 받아 두는 이름
    print(f"α = {값:.1f}   Recall@10 = {점수:.4f}")

알파 = 0.1                                       # 실험이 끝났으니 원래 값으로 되돌린다

α = 0.0   Recall@10 = 0.3918
α = 0.1   Recall@10 = 0.3892
α = 0.3   Recall@10 = 0.3725


α = 0.5   Recall@10 = 0.3377


α = 1.0   Recall@10 = 0.2463

**결과 읽는 법**

1. **α = 0 이 가장 높습니다.** 조금만 섞어도(α = 0.1) 점수가 내려갑니다.

2. **왜 오르지 않을까**
   - 「레버리지를 담는 사람은 레버리지를 또 담는다」는 점은 **4주차 잠재 요인이 이미 거래 기록에서 배웠습니다.**
   - 상품 정보는 모델에게 **새 정보가 아니었습니다.** 이미 반영된 정보를 한 번 더 더하면 새로 알게 되는 것은 없고, 상품 정보가 비슷한 종목의 점수만 올라갑니다. 이런 정보를 **중복 정보(redundant information)** 라고 합니다.

3. **그럼 상품 정보는 쓸모가 없을까**
   - 점수를 올리는 데는 쓸모가 없었습니다.
   - 대신 6.2 에서 본 문제 — **안정형에게 위험도 5** — 를 막는 데 씁니다. 다음 절에서 다룹니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.6 적합성 점검 — 성향에 맞지 않는 추천 세기</mark>

**6.6 에서 하는 일** — 300명에게 준 추천 3,000개 가운데 **투자 성향에 맞지 않는 것**이 몇 개인지 셉니다.

**1. 적합성원칙**
- 금융소비자보호법 제17조 ③ — 「… 그 일반금융소비자에게 적합하지 아니하다고 인정되는 계약 체결을 권유해서는 아니 된다」.
- 추천 모델은 모든 고객에게 상품을 권하는 셈이므로 **금융 회사라면 이 기준을 신경 써야 합니다.**

**2. 오늘 쓰는 기준 (이 수업에서 정한 예시)**

| 투자 성향 | 추천해도 되는 위험도 |
|---|---|
| 안정형 | 3 까지 |
| 성장형 | 4 까지 |
| 공격형 | 5 까지 (제한 없음) |

- 실제 기준은 회사마다 다릅니다. 오늘은 이 표를 **지켜야 할 기준**으로 삼습니다.

In [17]:
# 이 셀에서 하는 일 — 추천 3,000개 중 성향 기준을 넘는 것을 센다
# 왜 하나 — 6.2 에서 U0001 한 명으로 본 문제가 전체로는 얼마나 큰지 잰다

적합_기준 = {"안정형": 3, "성장형": 4, "공격형": 5}    # 딕셔너리 {성향: 추천해도 되는 최대 위험도}

def 안_맞는_개수(추천함수):                        # 추천 함수 자체를 받는다
    """300명의 추천 10개씩을 보고, 성향 기준을 넘는 종목 수를 센다."""
    개수 = 0                                       # 센 개수. 0 부터 시작한다
    for 사람 in users["user_id"]:                  # 300명을 한 명씩
        for 번호 in 추천함수(사람, 이미_담은것.get(사람, set())):   # get = 없으면 빈 집합
            if 위험도[번호] > 적합_기준[성향[사람]]:              # > = 크다. 기준보다 크면 센다
                개수 = 개수 + 1                    # 하나 더 센다
    return 개수                                    # 추천 3,000개 중 기준을 넘은 개수

print(f"4주차 추천 3,000개 중 성향에 맞지 않는 것 : {안_맞는_개수(지난주_추천)}개")

4주차 추천 3,000개 중 성향에 맞지 않는 것 : 205개


**결과 읽는 법**

- 300명에게 10개씩 준 4주차 추천 3,000개 가운데 적합 기준을 넘는 추천의 개수입니다.
  - **205개**가 투자 성향의 기준을 넘었습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.7 위험도 상한 — 후처리 필터링으로 추천 거르기</mark>

**6.7 에서 하는 일** — 4주차 추천 순서에서 **상한을 넘는 종목을 빼고**, 남은 종목 중 위에서부터 10개를 추천합니다.

**1. 모델은 그대로입니다**
- 점수 계산은 4주차와 똑같습니다. **마지막에 걸러 내기만 합니다.**
- 빠진 자리는 **그다음 순위 종목**이 채웁니다. 그래서 추천은 언제나 10개입니다.

**2. 신규 15명도 똑같이 거릅니다**
- 신규 고객은 전체 인기 목록을 받는데, 인기 상위 10개에도 위험도 4 종목이 세 개 있습니다(Taiwan Semiconductor · Invesco QQQ Trust · KODEX 미국나스닥100).
  - 안정형 신규 고객에게는 기준(3 까지)을 넘는 종목입니다.
  - 투자 성향은 가입할 때 받으므로 신규 고객에게도 상한을 걸 수 있습니다.

**3. 상한과 기준을 따로 둔 이유**
- `적합_기준` — **지켜야 할 기준.** 6.6 에서 정했고 바꾸지 않습니다.
- `상한` — **추천 목록에 실제로 거는 값.** 6.8 에서 여러분이 바꿔 봅니다.

In [18]:
# 이 셀에서 하는 일 — 4주차 추천 순서에서 상한을 넘는 종목을 빼고 10개를 준다
# 왜 하나 — 모델을 고치지 않고 마지막에 거르기만 해서 성향에 맞지 않는 추천을 없앤다

상한 = {"안정형": 3, "성장형": 4, "공격형": 5}         # 추천 목록에 거는 상한 — 6.8 에서 바꿔 본다

def 상한_추천(사람, 이미):                               # 4주차 추천 순서에서 상한을 넘는 종목만 빼는 함수
    """4주차 추천 순서에서 성향의 상한을 넘는 종목을 빼고 위에서 10개."""
    순서 = 전체_인기순위                                 # 신규는 인기 목록에서 시작한다
    if 사람 in 사용자_목록:                              # 기존 투자자는 4주차 점수 순서
        순서 = 점수순으로_종목_정렬(사용자_요인[사용자_목록.index(사람)] @ 아이템_요인)   # 6.1 과 같은 계산
    통과 = []                                            # 상한을 통과한 종목을 담을 빈 목록
    for 번호 in 순서:                                    # 점수가 높은 종목부터 하나씩
        if 위험도[번호] <= 상한[성향[사람]]:              # <= 작거나 같다. 상한 이하만 남긴다
            통과.append(번호)                            # 통과하면 목록에 붙인다
    return recsys.take(통과, 이미)                       # 통과한 것 중 이미 담은 것을 빼고 10개

In [19]:
# 이 셀에서 하는 일 — 상한을 건 추천을 채점하고, 성향에 맞지 않는 추천을 다시 센다
# 왜 하나 — 무엇을 얻었고(안 맞는 추천 수) 무엇을 내줬는지(점수) 함께 본다

상한_점수, _ = recsys.recall_at_k(상한_추천, train, test)

print(f"4주차 그대로   — Recall@10 = {지난주_점수:.4f} · 성향에 맞지 않는 추천 {안_맞는_개수(지난주_추천)}개")
print(f"위험도 상한    — Recall@10 = {상한_점수:.4f} · 성향에 맞지 않는 추천 {안_맞는_개수(상한_추천)}개")
print(f"→ 점수 {상한_점수 - 지난주_점수:+.4f}")          # :+.4f = 부호까지 붙여 소수 넷째 자리

4주차 그대로   — Recall@10 = 0.3918 · 성향에 맞지 않는 추천 205개
위험도 상한    — Recall@10 = 0.3899 · 성향에 맞지 않는 추천 0개
→ 점수 -0.0019


**결과 읽는 법**

- 첫 줄은 4주차 추천 그대로입니다.
  - 300명에게 준 추천 3,000개 중 **205개**가 투자 성향의 기준을 넘었습니다.
- 둘째 줄은 같은 추천에 위험도 상한을 건 결과입니다.
  - 기준을 넘는 추천이 **0개**가 되었습니다.
  - Recall@10은 0.3918 → **0.3899**입니다.
- 마지막 줄 `-0.0019`는 상한을 걸어서 **내준 점수**입니다.
  - 205개를 걸러 내는 대가로 Recall@10을 0.0019만큼 잃었다는 뜻입니다.

**🗣 수업에서 함께 보기**

위험도 상한으로 성향에 맞지 않는 추천을 **205개 → 0개**로 줄였는데, Recall@10 은 0.3918 → 0.3899 로 **0.0019 만** 내려갔습니다.

추천 목록에서 종목이 많이 빠졌는데, 왜 점수는 거의 그대로일까요?

- 힌트 — 6.7 의 「빠진 자리는 그다음 순위 종목이 채웁니다」를 다시 읽어 보세요. 빠진 종목이 **원래 정답이었는지**가 열쇠입니다.
- 수업에서 함께 이야기합니다. 답은 노트북 맨 아래 **「답」**에도 있습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.8 상한 정하기 — 점수와 안 맞는 추천 수 비교</mark>

**6.8 에서 하는 일** — 상한을 여러 값으로 바꿔 보고, **점수와 성향에 맞지 않는 추천 수**를 나란히 비교합니다.

**1. 무엇을 보고 고를까**
- 상한을 **느슨하게** 걸면 → 점수는 지키지만 기준을 넘는 추천이 남습니다.
- 상한을 **세게** 걸면 → 기준은 지키지만 고객이 실제로 담을 종목까지 빠져 점수가 내려갑니다.

**2. 답할 질문**
- 성향에 맞지 않는 추천을 **0개**로 만들면서 점수를 **가장 덜 내주는** 상한은 무엇일까요?

In [20]:
# 이 셀에서 하는 일 — 상한을 바꿔 가며 점수와 안 맞는 추천 수를 함께 잰다
# 왜 하나 — 기준을 지키는 대가로 점수를 얼마나 내주는지 보고 상한을 고른다

실험 = [{"안정형": 5, "성장형": 5, "공격형": 5},       # 상한 없음 = 4주차
        {"안정형": 4, "성장형": 5, "공격형": 5},       # 안정형의 위험도 5 만 뺀다
        {"안정형": 3, "성장형": 4, "공격형": 5},       # 6.7 의 기본값
        {"안정형": 2, "성장형": 3, "공격형": 4}]       # 기준보다 한 단계씩 더 세게

for 후보 in 실험:                                    # 상한 네 가지를 하나씩
    상한 = 후보                                        # 위 함수가 이 값을 보고 거른다
    점수, _ = recsys.recall_at_k(상한_추천, train, test)   # 6.7 과 같은 채점
    print(f"안정 {후보['안정형']} · 성장 {후보['성장형']} · 공격 {후보['공격형']}   Recall@10 {점수:.4f} · 안 맞는 추천 {안_맞는_개수(상한_추천):3d}개")

상한 = {"안정형": 3, "성장형": 4, "공격형": 5}         # 실험이 끝났으니 기본값으로 되돌린다

안정 5 · 성장 5 · 공격 5   Recall@10 0.3918 · 안 맞는 추천 205개
안정 4 · 성장 5 · 공격 5   Recall@10 0.3862 · 안 맞는 추천 154개
안정 3 · 성장 4 · 공격 5   Recall@10 0.3899 · 안 맞는 추천   0개


안정 2 · 성장 3 · 공격 4   Recall@10 0.2688 · 안 맞는 추천   0개


**결과 읽는 법**

1. **첫 줄** — 상한이 없습니다. 점수는 4주차와 같고, 성향에 맞지 않는 추천이 그대로 남습니다.

2. **둘째 줄** — 안정형에게서 위험도 5 만 뺐습니다. 아직 기준을 넘는 추천이 남습니다.
   - 점수가 셋째 줄보다 조금 낮은 것은 빠진 자리를 채운 종목이 정답이었는지에 따라 점수가 오르내리기 때문입니다. 0.004 정도는 작은 차이입니다.

3. **셋째 줄** — 상한을 기준과 같게 걸었습니다. **안 맞는 추천이 0개**가 되고, 점수는 0.3918 → 0.3899 로 **0.002 만** 내려갑니다.

4. **넷째 줄** — 기준보다 한 단계씩 더 세게 걸었습니다. 안 맞는 추천은 셋째 줄과 똑같이 0개인데 **점수만 더 내려갑니다.**

**한 줄로 정리하면** — 기준보다 세게 걸어도 더 지켜지는 것은 없고 점수만 깎입니다. **적합 기준과 같은 값을 상한으로 거는 것**이 점수를 가장 덜 내주는 선택입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.9 오늘의 마무리</mark>

1. 상품 정보 네 가지를 **원핫 인코딩**으로 0/1 열 41개로 펼쳤습니다.
2. 콘텐츠 기반 점수를 협업 필터링 점수에 **섞어 봤지만(하이브리드 추천) 점수가 오르지 않았습니다.** 모델이 거래 기록에서 이미 배운 정보였기 때문입니다.
3. 4주차 모델은 안정형 U0001 에게 **위험도 4~5 종목을 네 개** 추천했습니다.
4. 투자 성향별 **위험도 상한**을 걸어 성향에 맞지 않는 추천을 **205개에서 0개**로 줄였고, 그 대가로 점수를 **0.3918 → 0.3899** 로 조금 내줬습니다.

**오늘 기억할 한 줄** — **상품 정보는 점수를 올리는 데가 아니라 투자 성향에 맞지 않는 추천을 거르는 데 썼습니다.**

다음 주에는 지금까지 셀을 하나씩 눌러 한 일 — 모델 학습과 추천 — 을 **사람이 손대지 않아도 한 번에 돌아가도록** 묶고, 결과를 파일로 남깁니다.

In [21]:
# 이 셀에서 하는 일 — 오늘 점수를 점수판에 남긴다
# 왜 하나 — 점수가 내려간 주차도 그대로 적는다. 점수를 무엇과 바꿨는지 설명할 수 있으면 된다

recsys.record(6, "투자 성향별 위험도 상한", 상한_점수,
              note=f"4주차 추천에서 성향 기준을 넘는 종목을 뺐다. 안 맞는 추천 {안_맞는_개수(지난주_추천)}개 → 0개")

recsys.leaderboard(upto=6)   # 오늘까지 쌓인 점수판 (뒤 주차는 빼고 본다)

레벨 6 · 투자 성향별 위험도 상한 · Recall@10 = 0.3899


,level,name,recall_at_10,note
0,1,모두에게 같은 인기 순위,0.2163,알고리즘 없음. 인기 상위 10개를 모두에게 같게 추천했다
1,2,취향이 비슷한 세그먼트끼리,0.2684,거래 기록으로 주력 섹터와 평균 위험도를 뽑아 세그먼트로 나눴다
2,3,세그먼트별 목록 — 2주차와 같음,0.2684,추천 방식은 2주차와 같다. 무작위로 나누면 0.3150 이 나오는데 그것은 미래를...
3,4,행렬분해 SVD · 요인 8개,0.3918,규칙을 사람이 정하지 않았다. 사람 285 × 종목 100 표를 요인 8개로 쪼갰다
4,5,2단계 추천 · 후보 생성 + 순위 매기기,0.3382,2주차 세그먼트로 후보 40개를 추리고 그 40개만 4주차 모델로 정렬했다
5,6,투자 성향별 위험도 상한,0.3899,4주차 추천에서 성향 기준을 넘는 종목을 뺐다. 안 맞는 추천 205개 → 0개


**결과 읽는 법**

- 첫 줄은 오늘 점수를 점수판에 남겼다는 확인입니다.
  - 레벨 6 · 투자 성향별 위험도 상한 · Recall@10 = **0.3899** 입니다.
- 아래 표는 1주차부터 6주차까지의 점수판입니다.
  - `level` 은 주차, `name` 은 그 주의 추천 방식, `recall_at_10` 은 점수, `note` 는 한 줄 설명입니다.
  - `note` 끝의 `...` 은 글이 길어서 화면에서 잘린 것입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📝 혼자 해 보기</mark>

수업이 끝난 뒤 혼자 풀어 보는 문제입니다. 문제마다 **어느 절의 결과를 보면 되는지** 제목에 적어 두었습니다.

답은 바로 아래 **「답」**에 있습니다. 먼저 스스로 생각해 본 뒤에 읽어 보세요.

**📝 혼자 해 보기 1 · 6.4 결과**

콘텐츠 기반 추천의 Recall@10은 **0.2463**으로, 4주차(0.3918)보다 훨씬 낮습니다.

상품 정보만으로는 **무엇을 구별하지 못해서** 점수가 낮을까요?

- 힌트 — 6.3의 0/1 표에서 **삼성생명(I027)과 삼성화재(I028)의 줄**을 비교해 보세요.
- 답은 노트북 맨 아래 **「답」**에 있습니다.

**📝 혼자 해 보기 2 · 6.5 결과**

α = 1.0 의 Recall@10 은 **0.2463** 입니다. 6.4 의 콘텐츠 기반 추천 점수와 **소수점 넷째 자리까지 같습니다.**

점수를 표준화까지 했는데, 왜 똑같이 나올까요?

- 힌트 — 섞는 식 `(1 - α) × 협업 필터링 점수 + α × 콘텐츠 기반 점수` 에 α = 1.0 을 넣어 보세요. 그리고 표준화가 점수의 **순서**를 바꾸는지 생각해 보세요.
- 답은 노트북 맨 아래 **「답」**에 있습니다.

**📝 혼자 해 보기 3 · 6.8 결과**

회사가 기준을 바꿔 **「안정형은 위험도 2 까지」**로 더 엄격하게 정했다고 해 봅시다.

지금 상한(안정형 3 · 성장형 4 · 공격형 5)을 그대로 두면 어떻게 될까요? 상한을 어떻게 바꿔야 하고, 그 대가는 얼마일까요?

- 힌트 — 6.8 표에서 **상한 숫자와 「안 맞는 추천」 개수**가 어떻게 함께 움직이는지 보세요.
- 답은 노트북 맨 아래 **「답」**에 있습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">답</mark>

먼저 스스로 답을 생각해 본 뒤에 읽어 보세요. 답에 나오는 숫자는 모두 이 노트북을 실제로 실행해서 얻은 값입니다.

**함께 보기 답 — 빠진 종목 대부분이 원래 정답이 아니었고, 빈자리를 채운 종목이 그만큼 다시 맞혔기 때문입니다**

- 채점 대상 266명의 추천에서 상한 때문에 빠진 종목은 **179개**입니다.
  - 그중 실제로 다음 달에 담은 정답은 **15개**뿐입니다.
- 빠진 자리에는 그다음 순위 종목 **179개**가 새로 들어왔습니다.
  - 그중 정답도 **15개**입니다. 잃은 만큼 다시 맞힌 셈입니다.
- 맞힌 개수가 같은데도 점수가 0.0019 내려간 것은 Recall@10 의 계산 방식 때문입니다.
  - Recall@10 은 사람마다 「맞힌 개수 ÷ 그 사람의 정답 개수」를 구한 뒤 평균을 냅니다.
  - 그래서 같은 한 개라도 사람마다 점수에 주는 무게가 다르고, 잃은 쪽의 무게가 조금 더 컸습니다.
- **한 줄로 정리하면** — 성향에 맞지 않는 추천은 대부분 어차피 맞히지 못하는 추천이었고, 그래서 걸러 내도 점수를 거의 잃지 않았습니다.

**혼자 해 보기 1 답 — 상품 정보가 같은 종목끼리는 구별하지 못하기 때문입니다**

- 삼성생명과 삼성화재는 상품 정보 네 가지가 **완전히 같습니다.**
  - 둘 다 섹터 `금융` · 자산군 `주식` · 위험도 `3` · 테마 `배당`
  - 그래서 0/1 줄도 똑같고, U0001의 콘텐츠 기반 점수도 둘 다 **1.75**로 같습니다.
- 이런 종목이 생각보다 많습니다.
  - 종목은 100개인데, 서로 다른 상품 정보 조합은 **43가지**뿐입니다.
  - 가장 큰 묶음은 「지수 · ETF · 위험도 3 · 인덱스」로, **9개 종목**이 똑같은 줄을 가집니다.
- 콘텐츠 기반 추천은 이 9개 중 **어느 것을 담을지 알 수 없습니다.** 점수가 모두 같기 때문입니다.
  - 반대로 4주차 협업 필터링은 「비슷한 투자자들이 실제로 어느 것을 담았나」를 보기 때문에 이 9개를 구별합니다.
- **한 줄로 정리하면** — 상품 정보는 「어떤 종류를 좋아하나」까지만 알려 주고, 「그중 어느 종목인가」는 거래 기록이 알려 줍니다.

**혼자 해 보기 2 답 — α = 1.0 이면 콘텐츠 기반 점수만 남고, 표준화는 점수의 순서를 바꾸지 않기 때문입니다**

- α = 1.0 을 식에 넣으면 협업 필터링 점수 쪽이 **0 배**가 됩니다.
  - `(1 - 1.0) × 협업 필터링 점수 + 1.0 × 콘텐츠 기반 점수` = 콘텐츠 기반 점수
- 표준화는 한 사람의 점수 100개 모두에서 **같은 값(평균)을 빼고, 같은 값(표준편차)으로 나눕니다.**
  - 그래서 점수의 크기는 바뀌어도 **누가 몇 등인지는 그대로**입니다.
  - 6.5 의 함수 확인 출력에서 `1, 2, 3` 이 `-1.22, 0, 1.22` 가 되어도 순서는 그대로였습니다.
- 추천은 점수의 **순서**로 위에서 10개를 고릅니다.
  - 순서가 같으니 추천 10개가 같고, Recall@10 도 0.2463 으로 같습니다.
- **한 줄로 정리하면** — 표준화는 두 점수의 범위를 맞출 뿐, 한 점수 안의 순위는 바꾸지 않습니다.

**혼자 해 보기 3 답 — 지금 상한으로는 324개가 새 기준을 넘고, 상한을 2 · 4 · 5 로 낮추면 0개가 되는 대신 점수가 0.3704 로 내려갑니다**

- 상한을 그대로(안정형 3) 두면, 안정형에게 간 위험도 3 종목이 모두 새 기준을 넘게 됩니다.
  - 실제로 세어 보면 성향에 맞지 않는 추천이 **324개**가 됩니다.
- 상한도 기준에 맞춰 **안정형 2 · 성장형 4 · 공격형 5** 로 바꿔야 합니다.
  - 그러면 안 맞는 추천은 다시 **0개**가 됩니다.
  - 대신 Recall@10 이 0.3899 → **0.3704** 로 0.0195 내려갑니다. 6.8 의 실험에서 본 것처럼, 상한이 엄격할수록 고객이 실제로 담을 종목도 함께 빠지기 때문입니다.
- **한 줄로 정리하면** — 기준이 바뀌면 상한도 같이 바꿔야 하고, 기준이 엄격할수록 점수를 더 많이 내줍니다.